# Notebook 2 — Data Cleaning & Transformation

**Use Case 2 goal:** take the exploration-ready retail file and turn it into a cleaner, reporting-friendly dataset.

## What learners should understand
- Why data cleaning is separated from raw ingestion
- Why each transformation is performed
- How the cleaned file becomes the input for the ETL aggregation step


## Dependencies, AWS setup, and files used

### Python packages
- **boto3** — used to read the Use Case 1 handoff file from Amazon S3 and write the cleaned dataset back to S3
- **pandas** — used for learner-friendly transformation logic before we translate the ETL idea into Glue PySpark
- **io** — used to move CSV content between S3 and pandas without extra manual downloads
- **pathlib** — kept as a local rehearsal fallback


In [12]:
# AWS-friendly setup cell for SageMaker Studio or any Jupyter environment with AWS credentials.
from pathlib import Path
from io import BytesIO, StringIO
import os
import boto3
import pandas as pd

AWS_REGION = os.getenv('AWS_REGION', boto3.session.Session().region_name or 'ap-south-1')

# ✅ FIX: Set your actual bucket directly
S3_BUCKET = "usecase-etl-1"

# ✅ FIX: Disable fallback since you're using real S3
USE_LOCAL_FALLBACK = False

s3_client = boto3.client('s3', region_name=AWS_REGION)


def parse_s3_uri(uri: str):
    bucket, key = uri.replace('s3://', '', 1).split('/', 1)
    return bucket, key


def read_csv_aws_first(s3_uri: str, local_path: Path) -> pd.DataFrame:
    """Try S3 first, fallback to local if it fails."""
    try:
        bucket, key = parse_s3_uri(s3_uri)
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        print("✅ Reading from S3:", s3_uri)
        return pd.read_csv(BytesIO(obj['Body'].read()))
    except Exception as e:
        print(f"⚠️ S3 read failed: {e}")
        print("📂 Falling back to local file:", local_path)
        return pd.read_csv(local_path)


def write_csv_aws_first(df: pd.DataFrame, s3_uri: str, local_path: Path) -> None:
    """Write to S3, fallback to local if needed."""
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    try:
        bucket, key = parse_s3_uri(s3_uri)
        s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue().encode('utf-8'))
        print("✅ Written to S3:", s3_uri)
    except Exception as e:
        print(f"⚠️ S3 write failed: {e}")
        print("📂 Writing locally instead:", local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_text(csv_buffer.getvalue(), encoding='utf-8')


LOCAL_INPUT_PATH = Path('./retail_exploration_ready.csv')
LOCAL_OUTPUT_PATH = Path('./retail_cleaned.csv')

INPUT_S3_URI = "s3://usecase-etl-1/processed/retail_exploration_ready.csv"
#s3://usecase-etl-1/processed/retail_exploration_ready.csv
OUTPUT_S3_URI = "s3://usecase-etl-1/processed/retail_cleaned.csv"

print('Input source:', INPUT_S3_URI)
print('Output target:', OUTPUT_S3_URI)

Input source: s3://usecase-etl-1/processed/retail_exploration_ready.csv
Output target: s3://usecase-etl-1/processed/retail_cleaned.csv


## Step 1 — Read the exploration-ready file

### Why this step is performed
This notebook continues the same story from Use Case 1.

### Expected result
A DataFrame that is ready for formal cleaning rules.


In [13]:
df = read_csv_aws_first(INPUT_S3_URI, LOCAL_INPUT_PATH)
print('Shape before cleaning:', df.shape)
display(df.head())


✅ Reading from S3: s3://usecase-etl-1/processed/retail_exploration_ready.csv
Shape before cleaning: (500, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateParsed
0,536365,71053,WHITE METAL LANTERN,6,02/01/2011 11:08,5.49,17889.0,Belgium,2011-02-01 11:08:00
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,01/28/2011 11:32,4.22,16943.0,Germany,2011-01-28 11:32:00
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,02/05/2011 08:48,5.80,18065.0,Netherlands,2011-02-05 08:48:00
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,01/13/2011 13:54,7.55,14512.0,United Kingdom,2011-01-13 13:54:00
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,03/22/2011 17:56,4.03,17075.0,Germany,2011-03-22 17:56:00


## Step 2 — Convert dates and derive reporting fields

### Why this step is performed
Most analytics and ETL outputs require usable time fields such as transaction date, month, and year.


In [14]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['TransactionDate'] = df['InvoiceDate'].dt.date.astype(str)
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)

display(df[['InvoiceDate', 'TransactionDate', 'Year', 'Month']].head())


,InvoiceDate,TransactionDate,Year,Month
0,2011-02-01 11:08:00,2011-02-01,2011,2011-02
1,2011-01-28 11:32:00,2011-01-28,2011,2011-01
2,2011-02-05 08:48:00,2011-02-05,2011,2011-02
3,2011-01-13 13:54:00,2011-01-13,2011,2011-01
4,2011-03-22 17:56:00,2011-03-22,2011,2011-03


In [21]:
(df.columns)


Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'InvoiceDateParsed',
       'TransactionDate', 'Year', 'Month'],
      dtype='object')

## Step 3 — Handle missing values and remove invalid records

### Why this step is performed
Not every row should move forward into reporting. Here we apply simple, teachable business rules.


In [20]:
df['Description'] = df['Description'].fillna('UNKNOWN_ITEM')
df['CustomerID'] = df['CustomerID'].fillna('UNKNOWN_CUSTOMER')

df = df[df['InvoiceDate'].notna()]
df = df[df['UnitPrice'] > 0]

print('Shape after cleaning:', df.shape)


Shape after cleaning: (489, 12)


## Step 4 — Engineer business metrics

### Why this step is performed
Learners should see exactly where business fields such as revenue come from.


In [22]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['IsReturn'] = df['Quantity'] < 0

display(df[['Quantity', 'UnitPrice', 'Revenue', 'IsReturn']].head())


,Quantity,UnitPrice,Revenue,IsReturn
0,6,5.49,32.94,False
1,2,4.22,8.44,False
2,6,5.80,34.80,False
3,4,7.55,30.20,False
4,6,4.03,24.18,False


## Step 5 — Save the cleaned dataset

### Why this step is performed
This cleaned file becomes the ETL input for Notebook 3.


In [25]:
write_csv_aws_first(df, OUTPUT_S3_URI, LOCAL_OUTPUT_PATH)
print('Cleaned dataset saved to:')
print(OUTPUT_S3_URI if not USE_LOCAL_FALLBACK else LOCAL_OUTPUT_PATH.resolve())


✅ Written to S3: s3://usecase-etl-1/processed/retail_cleaned.csv
Cleaned dataset saved to:
s3://usecase-etl-1/processed/retail_cleaned.csv
